In [0]:
%python
spark.sql("USE CATALOG dbw_dp750_lab")
spark.sql("USE SCHEMA medallion_project")

df_silver = spark.table("dbw_dp750_lab.medallion_project.silver_taxi_trips")
df_silver.show(5)


+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|fare_amount|pickup_zip|dropoff_zip|trip_duration_minutes|
+--------------------+---------------------+-------------+-----------+----------+-----------+---------------------+
| 2016-02-13 21:47:53|  2016-02-13 21:57:15|          1.4|        8.0|     10103|      10110|    9.366666666666667|
| 2016-02-27 18:01:07|  2016-02-27 18:10:23|         2.83|       11.0|     10002|      11206|    9.266666666666667|
| 2016-02-28 15:16:44|  2016-02-28 15:35:32|          4.5|       16.5|     10282|      10069|                 18.8|
| 2016-02-04 00:22:49|  2016-02-04 00:38:19|          4.1|       15.0|     10019|      11104|                 15.5|
| 2016-02-20 19:28:12|  2016-02-20 20:08:33|         19.6|       54.5|     11422|      11238|                40.35|
+--------------------+---------------------+-------------+-----------+--

In [0]:
%python
from pyspark.sql import functions as F

df_gold = (
    df_silver
    .withColumn("trip_date", F.to_date("tpep_pickup_datetime"))
    .groupBy("trip_date", "pickup_zip")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_minutes"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance")
    )
    .orderBy("trip_date", "pickup_zip")
)

df_gold.show(10)
print(f"Gold row count: {df_gold.count()}")

+----------+----------+-----------+--------+--------------------+------------+
| trip_date|pickup_zip|total_trips|avg_fare|avg_duration_minutes|avg_distance|
+----------+----------+-----------+--------+--------------------+------------+
|2016-01-01|     10001|         19|    9.26|                9.89|        1.96|
|2016-01-01|     10002|         14|   13.21|               11.63|         3.5|
|2016-01-01|     10003|         18|    9.97|                9.93|        2.34|
|2016-01-01|     10005|          1|    16.0|               10.22|        5.02|
|2016-01-01|     10006|          1|    16.5|               14.28|         4.6|
|2016-01-01|     10007|          4|     9.0|                 9.3|        1.99|
|2016-01-01|     10009|         11|   13.86|               13.67|        3.65|
|2016-01-01|     10011|         12|   15.54|               11.47|        4.05|
|2016-01-01|     10012|         10|    8.85|                8.89|        1.99|
|2016-01-01|     10013|          5|     9.8|        

In [0]:
%python
df_gold.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

print("Gold table created successfully!")

Gold table created successfully!


In [0]:
%python
df_silver.filter(F.col("pickup_zip") == 10007) \
    .filter(F.col("trip_duration_minutes") > 60) \
    .select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_minutes", "fare_amount", "trip_distance") \
    .show(20, truncate=False)

+--------------------+---------------------+---------------------+-----------+-------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|fare_amount|trip_distance|
+--------------------+---------------------+---------------------+-----------+-------------+
+--------------------+---------------------+---------------------+-----------+-------------+



In [0]:
%sql
DESCRIBE DETAIL gold_daily_zip_summary;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-8647786562998471>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'DESCRIBE DETAIL gold_daily_zip_summary;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activity_logge

In [0]:
%sql
USE CATALOG dbw_dp750_lab;
USE SCHEMA medallion_project;

DESCRIBE DETAIL gold_daily_zip_summary;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,cd6a86a7-4a30-4ec4-97c3-a1d1a8da94fa,dbw_dp750_lab.medallion_project.gold_daily_zip_summary,null,abfss://unity-catalog-storage@dbstoragegmyhhr44z5qcw.dfs.core.windows.net/7405619560832544/__unitystorage/catalogs/f4b588cb-bdc8-42ba-a1d3-97f19b2ea8f5/tables/892dce16-3b14-4832-a6af-8c85012bc042,2026-07-28T19:32:49.507Z,2026-07-28T19:36:44.000Z,List(),List(),1,29140,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%python
df_gold_repartitioned = spark.table("gold_daily_zip_summary")

df_gold_repartitioned.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("trip_date") \
    .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

print("Table repartitioned by trip_date")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4627998061824264>, line 6
      1 df_gold_repartitioned = spark.table("gold_daily_zip_summary")
      3 df_gold_repartitioned.write.format("delta") \
      4     .mode("overwrite") \
      5     .partitionBy("trip_date") \
----> 6     .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")
      8 print("Table repartitioned by trip_date")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/

In [0]:
%python
df_gold_repartitioned = spark.table("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

df_gold_repartitioned.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("trip_date") \
    .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

print("Table repartitioned by trip_date")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4627998061824265>, line 6
      1 df_gold_repartitioned = spark.table("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")
      3 df_gold_repartitioned.write.format("delta") \
      4     .mode("overwrite") \
      5     .partitionBy("trip_date") \
----> 6     .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")
      8 print("Table repartitioned by trip_date")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /d

In [0]:
%python
df_gold_repartitioned = spark.table("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

df_gold_repartitioned.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("trip_date") \
    .saveAsTable("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

print("Table repartitioned by trip_date")

Table repartitioned by trip_date


In [0]:
%python
display(spark.sql("DESCRIBE DETAIL dbw_dp750_lab.medallion_project.gold_daily_zip_summary").select("numFiles", "sizeInBytes", "partitionColumns"))

numFiles,sizeInBytes,partitionColumns
60,172151,List(trip_date)


In [0]:
%sql
OPTIMIZE dbw_dp750_lab.medallion_project.gold_daily_zip_summary
ZORDER BY (pickup_zip);

path,metrics
abfss://unity-catalog-storage@dbstoragegmyhhr44z5qcw.dfs.core.windows.net/7405619560832544/__unitystorage/catalogs/f4b588cb-bdc8-42ba-a1d3-97f19b2ea8f5/tables/892dce16-3b14-4832-a6af-8c85012bc042,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 60, List(minCubeSize(107374182400), List(0, 0), List(60, 172151), 0, List(0, 0), 0, null), null, 0, 0, 60, 60, false, 0, 0, 1785433428134, 1785433432313, 8, 0, null, List(0, 0), null, 6, 6, 0, 0, null, null)"


In [0]:
%sql
VACUUM dbw_dp750_lab.medallion_project.gold_daily_zip_summary;

path
abfss://unity-catalog-storage@dbstoragegmyhhr44z5qcw.dfs.core.windows.net/7405619560832544/__unitystorage/catalogs/f4b588cb-bdc8-42ba-a1d3-97f19b2ea8f5/tables/892dce16-3b14-4832-a6af-8c85012bc042


In [0]:
%python
import time

df = spark.table("dbw_dp750_lab.medallion_project.gold_daily_zip_summary")

start = time.time()
df.filter("pickup_zip = 10001").show()
print(f"First run: {time.time() - start:.3f} seconds")

start = time.time()
df.filter("pickup_zip = 10001").show()
print(f"Second run (cached): {time.time() - start:.3f} seconds")

+----------+----------+-----------+--------+--------------------+------------+
| trip_date|pickup_zip|total_trips|avg_fare|avg_duration_minutes|avg_distance|
+----------+----------+-----------+--------+--------------------+------------+
|2016-01-22|     10001|         24|    10.0|               10.02|        2.02|
|2016-02-12|     10001|         25|    10.0|               12.94|        2.67|
|2016-02-14|     10001|         27|    10.0|               13.32|        2.56|
|2016-01-09|     10001|         19|    10.0|               10.14|        1.65|
|2016-01-30|     10001|         24|    10.0|               12.38|         2.3|
|2016-02-21|     10001|         26|    10.0|               12.98|        2.82|
|2016-01-16|     10001|         21|    10.0|               11.45|        2.09|
|2016-01-01|     10001|         19|    10.0|                9.89|        1.96|
|2016-02-05|     10001|         32|    10.0|               11.32|        2.16|
|2016-01-10|     10001|         12|    10.0|        